# Breast Cancer Segmentation – Semi Supervised AI System 
## AI Project SS2026

This project develops a **semi-supervised AI system** for automatic tumor segmentation in breast MRI images.

**Problem:** Manual tumor segmentation is time-consuming and requires expert radiologists,  
leaving most medical data unlabeled and unusable for supervised learning.

**Goal:** A semi-supervised pipeline that:
- Learns tumor patterns from labeled MRI scans
- Automatically generates **pseudo-ROI masks** for unlabeled data
- Evaluates results using segmentation metrics & visual comparison

**Workflow:** Data acquisition → Preprocessing → EDA → Visualization → Model Training → Evaluation

**Data Repository**
The dataset **"Advanced-MRI-Breast-Lesions"** ([doi.org/10.7937/C7X1-YN57](https://doi.org/10.7937/C7X1-YN57)) 
is from the public database *The Cancer Imaging Archive*. Dataset: 620+ breast MRI cases, ~99 with annotated ROI tumor masks
Due to its size of ~600 GB, it is currently stored on an external SSD.

**Code Repository**
[github.com/MelisaAta/AI_Project](https://github.com/MelisaAta/AI_Project.git)


## Preprocessing of the dataset

#### Main subfolder used from the patients folders: "Registered AX Sen Vibrant C - Series" (Best input sequence - registered post-contrast)

In [1]:
# =============================================================================
# Preprocess Dataset:
#
# What it does, per patient:
#   1. Selects the Registered AX Sen Vibrant series (raw Vibrant as fallback).
#   2. Loads + sorts slices by Z (ImagePositionPatient[2]).
#   3. Normalizes with a BODY-VOXEL window  -> fixes the washed-out registered
#      look (the constant dark-fill floor no longer drags percentile(1) down).
#   4. If the patient has a tumour ROI (DICOM-SEG), aligns each mask frame to
#      the nearest image slice by Z (tolerance Z_TOLERANCE) and saves it
#      alongside the image. Patients without a usable ROI get an all-zero mask
#      and has_roi = False.
#   5. Resizes to IMAGE_SIZE, stores images as float16 + masks as uint8.
#
# Output:
#   - chunk_XXXX.pkl   : BLOCK_SIZE patients per file (RAM-safe)
#   - manifest.csv     : one row per patient (chunk, has_roi, n_slices, status)
#
# Labeled vs unlabeled are kept in ONE stream, distinguished by a per-slice
# `has_roi` flag (better than separate folders such as labeled and unlabeld patients: lets you split train/val/test
# per patient across both groups without leakage). Filter on has_roi to get the supervised subset at train time.
#
# Resumable: re-running skips patients already in manifest.csv and continues the chunk numbering, so a crash/interrupt never forces a full redo.
# ==========================================================================================================================================================

import os
import gc
import csv
import time
import pickle
import numpy as np
import pydicom

try:
    import cv2
    def _resize(a, size, nearest=False):
        return cv2.resize(a, (size[1], size[0]),
                          interpolation=cv2.INTER_NEAREST if nearest else cv2.INTER_AREA)
except ImportError:                       # fallback if opencv isn't installed
    from scipy.ndimage import zoom
    def _resize(a, size, nearest=False):
        zy, zx = size[0] / a.shape[0], size[1] / a.shape[1]
        return zoom(a, (zy, zx), order=0 if nearest else 1)

# =============================================================================
# CONFIGURATION  — edit paths here
# =============================================================================
DICOM_FOLDER = r"D:\Advanced-MRI-Breast-Lesions\DICOM Images\manifest-1713182663002\Advanced-MRI-Breast-Lesions"
OUT_DIR      = r"D:\Advanced-MRI-Breast-Lesions\data\chunks_semisup"
MANIFEST     = os.path.join(OUT_DIR, "manifest.csv")

IMAGE_SIZE   = (256, 256)     # (H, W) — final slice size
Z_TOLERANCE  = 0.5            # mm — max image<->mask Z distance for a match
BLOCK_SIZE   = 8              # patients per chunk file (raise if you have RAM)
P_LOW, P_HIGH = 1.0, 99.0     # contrast window percentiles (body voxels)
EXCLUDE_AIR  = False          # True = also drop air for max contrast (rarely needed)

DROP_AIR     = True           # skip near-empty UNLABELED slices (saves disk)
AIR_FRACTION = 0.995          # slice is "air" if >99.5% of pixels < 0.02
# labeled slices are ALWAYS kept (empty masks on non-lesion slices are valid
# negatives for supervised training)

os.makedirs(OUT_DIR, exist_ok=True)


# =============================================================================
# SERIES SELECTION  (registered preferred, raw Vibrant as fallback)
# =============================================================================
def is_registered_ax_vibrant(name):
    n = name.upper()
    return ("REGISTERED" in n and "AX" in n and "VIBRAN" in n
            and "MULTIPHASE" not in n and "T2" not in n and "MASK" not in n)

def is_raw_ax_vibrant(name):
    n = name.upper()
    return ("AX" in n and "VIBRAN" in n and "REGISTERED" not in n
            and "MULTIPHASE" not in n and "TRAM" not in n and "MASK" not in n)

def _series_number(name):
    try:
        return float(name.split(".")[0])
    except ValueError:
        return float("inf")

def select_series_folder(patient_subfolder, mode="auto"):
    """mode = 'registered' | 'raw' | 'auto' (registered, else raw)."""
    dirs = [d for d in os.listdir(patient_subfolder)
            if os.path.isdir(os.path.join(patient_subfolder, d))]
    reg = sorted([d for d in dirs if is_registered_ax_vibrant(d)], key=_series_number)
    raw = sorted([d for d in dirs if is_raw_ax_vibrant(d)],        key=_series_number)
    if mode in ("registered", "auto") and reg:
        return os.path.join(patient_subfolder, reg[0]), reg[0], "registered"
    if mode in ("raw", "auto") and raw:
        return os.path.join(patient_subfolder, raw[0]), raw[0], "raw"
    return None, None, None


# =============================================================================
# ORIENTATION + NORMALIZATION  (must match the verification script exactly)
# =============================================================================
def orient(x):
    """180 deg rotation == np.fliplr(np.flipud(x)). Apply to BOTH image and mask."""
    return np.rot90(x, 2)

def normalize_volume(vol, p_low=1.0, p_high=99.0, exclude_air=False):
    """
    Global percentile windowing with bounds estimated from BODY voxels only.
    Registered series fill out-of-FOV regions with a constant below the air
    background; that floor drags percentile(1) far down and the image turns
    foggy. Dropping that floor (and optionally air) restores crisp contrast.
    """
    v = np.nan_to_num(vol.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    vmin, vmax = float(v.min()), float(v.max())
    eps  = (vmax - vmin) * 1e-3
    body = v[v > vmin + eps]                       # drop constant dark-fill plateau
    if exclude_air and body.size:
        thr = 0.10 * np.percentile(body, 99.5)
        body2 = body[body > thr]
        if body2.size > 1000:
            body = body2
    if body.size < 100:
        body = v.ravel()
    lo, hi = np.percentile(body, p_low), np.percentile(body, p_high)
    if hi - lo < 1e-6:
        lo, hi = vmin, vmax
    return np.clip((v - lo) / (hi - lo + 1e-6), 0.0, 1.0)


# =============================================================================
# DICOM LOADING + ROI ALIGNMENT
# =============================================================================
def _read_slices(series_folder):
    """Return [(z, 2D float32 array), ...] sorted by Z."""
    recs = []
    for f in os.listdir(series_folder):
        if not f.lower().endswith(".dcm"):
            continue
        try:
            ds  = pydicom.dcmread(os.path.join(series_folder, f))
            arr = ds.pixel_array.astype(np.float32)
            if arr.ndim == 3:                     # packed multi-frame safety
                arr = arr[arr.shape[0] // 2]
            z = float(ds.ImagePositionPatient[2]) if hasattr(ds, "ImagePositionPatient") else 0.0
            recs.append((z, arr))
        except Exception:
            continue
    recs.sort(key=lambda r: r[0])
    return recs

def find_roi_file(patient_subfolder):
    rois = [d for d in os.listdir(patient_subfolder)
            if os.path.isdir(os.path.join(patient_subfolder, d))
            and "ROI" in d.upper() and "TRAM" not in d.upper()]
    for d in rois:
        folder = os.path.join(patient_subfolder, d)
        for f in sorted(os.listdir(folder)):
            if f.lower().endswith(".dcm"):
                return os.path.join(folder, f), d
    return None, None

def load_seg_aligned(roi_file, image_zs, image_hw, z_tol=0.5):
    """Build a mask volume aligned to the image slices by nearest Z."""
    ds_seg  = pydicom.dcmread(roi_file)
    seg_arr = ds_seg.pixel_array.astype(np.uint8)
    if seg_arr.ndim == 2:
        seg_arr = seg_arr[None, ...]
    try:
        seg_zs = [float(fg.PlanePositionSequence[0].ImagePositionPatient[2])
                  for fg in ds_seg.PerFrameFunctionalGroupsSequence]
    except Exception:
        try:
            sg = ds_seg.SharedFunctionalGroupsSequence[0]
            z0 = float(sg.PlanePositionSequence[0].ImagePositionPatient[2])
            seg_zs = [z0] * seg_arr.shape[0]
        except Exception as e:
            raise RuntimeError(f"Could not read per-frame Z from SEG: {e}")
    seg_zs = np.array(seg_zs, dtype=float)

    z_to_mask = {}                                # union frames sharing a Z (multi-lesion)
    for i, z in enumerate(seg_zs):
        key = round(float(z), 2)
        frame = seg_arr[i] > 0
        z_to_mask[key] = (z_to_mask[key] | frame) if key in z_to_mask else frame
    uniq_z = np.array(sorted(z_to_mask.keys()))

    H, W = int(image_hw[0]), int(image_hw[1])
    aligned, dists = np.zeros((len(image_zs), H, W), np.uint8), []
    for i, z in enumerate(image_zs):
        j = int(np.argmin(np.abs(uniq_z - z)))
        d = abs(uniq_z[j] - z)
        if d <= z_tol:
            m = z_to_mask[uniq_z[j]].astype(np.uint8)
            if m.shape != (H, W):
                m = (_resize(m, (H, W), nearest=True) > 0).astype(np.uint8)
            aligned[i] = (m > 0).astype(np.uint8)
            dists.append(d)

    n_matched = int((aligned.reshape(len(aligned), -1).sum(1) > 0).sum())
    return aligned, n_matched


# =============================================================================
# PER-PATIENT PROCESSING
# =============================================================================
def process_patient(pid):
    """Return dict(images, masks, has_roi, ...) or None if unusable."""
    p_path = os.path.join(DICOM_FOLDER, pid)
    subs = [d for d in os.listdir(p_path) if os.path.isdir(os.path.join(p_path, d))]
    if not subs:
        return None
    psub = os.path.join(p_path, subs[0])

    series_folder, sname, kind = select_series_folder(psub, mode="auto")
    if series_folder is None:
        return None

    recs = _read_slices(series_folder)
    if not recs:
        return None
    zs  = np.array([r[0] for r in recs])
    vol = np.stack([r[1] for r in recs], axis=0)
    vol = normalize_volume(vol, P_LOW, P_HIGH, EXCLUDE_AIR)   # SAME as verification

    # --- ROI (if present) ---
    roi_file, _ = find_roi_file(psub)
    has_roi = roi_file is not None
    n_matched = 0
    if has_roi:
        try:
            mask_vol, n_matched = load_seg_aligned(roi_file, zs, vol.shape[1:], Z_TOLERANCE)
            if n_matched == 0:               # ROI exists but nothing aligned -> unlabeled
                has_roi = False
        except Exception:
            has_roi = False
    if not has_roi:
        mask_vol = np.zeros_like(vol, dtype=np.uint8)

    # --- resize + filter + orient (image & mask identically) ---
    H, W = IMAGE_SIZE
    imgs, msks = [], []
    for i in range(vol.shape[0]):
        img, msk = vol[i], mask_vol[i]
        if DROP_AIR and msk.sum() == 0 and (img < 0.02).mean() > AIR_FRACTION:
            continue                          # drop background-only UNLABELED slices
        img_r = _resize(img, (H, W), nearest=False)
        msk_r = (_resize(msk, (H, W), nearest=True) if msk.max() > 0
                 else np.zeros((H, W), np.uint8))
        imgs.append(orient(img_r).astype(np.float16))
        msks.append((orient(msk_r) > 0).astype(np.uint8))
    if not imgs:
        return None

    return dict(
        pid=pid, sname=sname, kind=kind, has_roi=has_roi, n_matched=n_matched,
        images=np.stack(imgs)[..., None],         # (n, H, W, 1) float16
        masks=np.stack(msks)[..., None],          # (n, H, W, 1) uint8
    )


# =============================================================================
# MAIN LOOP  (resume + chunked flush + live progress)
# =============================================================================
def main():
    # ---- resume: which patients are already done? ----
    processed = set()
    if os.path.exists(MANIFEST):
        with open(MANIFEST, newline="") as f:
            processed = {row["patient_id"] for row in csv.DictReader(f)}
    else:
        with open(MANIFEST, "w", newline="") as f:
            csv.writer(f).writerow(
                ["patient_id", "chunk", "kind", "has_roi", "n_lesion_slices",
                 "n_slices", "status"])

    all_patients = sorted([d for d in os.listdir(DICOM_FOLDER)
                           if os.path.isdir(os.path.join(DICOM_FOLDER, d))])
    todo = [p for p in all_patients if p not in processed]
    chunk_idx = len([f for f in os.listdir(OUT_DIR)
                     if f.startswith("chunk_") and f.endswith(".pkl")])

    print("=" * 64)
    print(f"Total patients     : {len(all_patients)}")
    print(f"Already processed  : {len(processed)}")
    print(f"To process now     : {len(todo)}")
    print(f"Next chunk index   : {chunk_idx:04d}")
    print(f"Output folder      : {OUT_DIR}")
    print("=" * 64)

    buf = {"images": [], "masks": [], "pids": [], "hasroi": []}
    rows, n_block = [], 0
    tot_slices = tot_labeled = tot_lesion = n_ok = n_skip = n_roi = 0
    t0 = time.time()

    def flush(idx):
        if not buf["images"]:
            return idx
        chunk = {
            "images":      np.concatenate(buf["images"], 0),   # (N, H, W, 1) float16
            "masks":       np.concatenate(buf["masks"], 0),    # (N, H, W, 1) uint8
            "patient_ids": np.concatenate(buf["pids"], 0),     # (N,) per slice
            "has_roi":     np.concatenate(buf["hasroi"], 0),   # (N,) bool per slice
        }
        path = os.path.join(OUT_DIR, f"chunk_{idx:04d}.pkl")
        with open(path, "wb") as f:
            pickle.dump(chunk, f, protocol=4)
        with open(MANIFEST, "a", newline="") as f:
            csv.writer(f).writerows(rows)
        mb = chunk["images"].nbytes / 1e6 + chunk["masks"].nbytes / 1e6
        print(f"  -> wrote chunk_{idx:04d}.pkl  "
              f"slices={chunk['images'].shape[0]:>5}  "
              f"labeled={int(chunk['has_roi'].sum()):>5}  ({mb:.0f} MB)")
        for k in buf:
            buf[k].clear()
        rows.clear()
        del chunk
        gc.collect()
        return idx + 1

    for n, pid in enumerate(todo, 1):
        try:
            rec = process_patient(pid)
        except Exception as e:
            rec = None
            print(f"  [{n}/{len(todo)}] {pid}: ERROR {e}")

        if rec is None:
            n_skip += 1
            rows.append([pid, "-", "-", 0, 0, 0, "skipped"])
            continue

        k = rec["images"].shape[0]
        lesion = int(rec["masks"].reshape(k, -1).sum(1).astype(bool).sum())
        buf["images"].append(rec["images"])
        buf["masks"].append(rec["masks"])
        buf["pids"].append(np.array([pid] * k))
        buf["hasroi"].append(np.array([rec["has_roi"]] * k, dtype=bool))
        rows.append([pid, f"chunk_{chunk_idx:04d}.pkl", rec["kind"],
                     int(rec["has_roi"]), lesion, k, "ok"])

        n_ok += 1
        tot_slices += k
        if rec["has_roi"]:
            n_roi += 1
            tot_labeled += k
            tot_lesion += lesion
        n_block += 1

        # live progress every patient
        elapsed = time.time() - t0
        rate = n / elapsed if elapsed else 0
        eta = (len(todo) - n) / rate if rate else 0
        flag = "ROI" if rec["has_roi"] else " - "
        print(f"  [{n:>4}/{len(todo)}] {pid:<12} {flag}  "
              f"slices={k:>3} lesion={lesion:>3}  "
              f"({rate:.1f} pt/s, ETA {eta/60:.1f} min)", end="\r")

        if n_block >= BLOCK_SIZE:
            print()                              # newline before chunk message
            chunk_idx = flush(chunk_idx)
            n_block = 0

    print()
    chunk_idx = flush(chunk_idx)                 # final partial block

    print("=" * 64)
    print("DONE")
    print(f"  patients processed : {n_ok}   (skipped {n_skip})")
    print(f"  patients with ROI  : {n_roi}")
    print(f"  total slices saved : {tot_slices:,}")
    print(f"    labeled slices   : {tot_labeled:,}")
    print(f"    lesion slices    : {tot_lesion:,}  (mask actually non-empty)")
    print(f"    unlabeled slices : {tot_slices - tot_labeled:,}")
    print(f"  chunks written     : up to chunk_{chunk_idx-1:04d}.pkl")
    print(f"  manifest           : {MANIFEST}")
    print("=" * 64)


if __name__ == "__main__":
    main()

Total patients     : 632
Already processed  : 0
To process now     : 632
Next chunk index   : 0000
Output folder      : D:\Advanced-MRI-Breast-Lesions\data\chunks_semisup
  [   8/632] AMBL-008     ROI  slices=116 lesion= 23  (0.3 pt/s, ETA 41.5 min)
  -> wrote chunk_0000.pkl  slices=  924  labeled=  692  (182 MB)
  [  16/632] AMBL-016     ROI  slices=116 lesion= 11  (0.2 pt/s, ETA 46.4 min)
  -> wrote chunk_0001.pkl  slices=  928  labeled=  580  (182 MB)
  [  24/632] AMBL-024     ROI  slices=116 lesion= 13  (0.2 pt/s, ETA 50.5 min)
  -> wrote chunk_0002.pkl  slices=  928  labeled=  464  (182 MB)
  [  33/632] AMBL-033     ROI  slices=116 lesion=  5  (0.2 pt/s, ETA 50.7 min)
  -> wrote chunk_0003.pkl  slices=  928  labeled=  696  (182 MB)
  [  41/632] AMBL-041      -   slices=116 lesion=  0  (0.2 pt/s, ETA 53.8 min)
  -> wrote chunk_0004.pkl  slices=  928  labeled=  464  (182 MB)
  [  50/632] AMBL-050     ROI  slices=116 lesion=  5  (0.2 pt/s, ETA 54.9 min)
  -> wrote chunk_0005.pkl  sli